In [ ]:
%pip install atproto 

In [ ]:
from atproto import Client
import requests
import time
from datetime import datetime, timedelta
import pandas as pd
import re
import os

#login into bluesky
client = Client()
client.login('USERNAME HERE', 'PASSWORD HERE')

#load posts from CSV
posts_df = pd.read_csv("posts_links.csv")

#iterate over each post link
for idx, row in posts_df.iterrows():
    link = row['link']
    print(f"\n{'='*50}")
    print(f"Processing post {idx + 1}/{len(posts_df)}: {link}")
    print('='*50)
    
    try:
        #extract handle from link (format: https://bsky.app/profile/handle/post/postid)
        match = re.search(r'bsky\.app/profile/([^/]+)/post/([^/]+)', link)
        if not match:
            print(f"Invalid link format: {link}")
            continue
        
        handle = match.group(1)
        post_id = match.group(2)
        
        #get DID from handle
        did = requests.get(
            "https://bsky.social/xrpc/com.atproto.identity.resolveHandle",
            params={"handle": handle}
        ).json()["did"]
        
        uri = f"at://{did}/app.bsky.feed.post/{post_id}"
        
        #get thread with retry
        for i in range(3):
            try:
                res = client.get_post_thread(uri=uri)
                break
            except Exception as e:
                print(f"retrying... {e}")
                time.sleep(2)
        
        thread = res.thread
        
        #collect all comments
        all_replies = []  
        for reply in thread.replies:
            reply_dt = datetime.fromisoformat(reply.post.record.created_at.replace("Z", "+00:00"))
            
            if not reply.post.record.text or reply.post.record.text.strip() == "":
                continue  #skips empty comments if there are bugs with extraction

            #add all comments
            all_replies.append({
                "text": reply.post.record.text,
                "time": reply_dt,
                "political_position": ""  #empty for manual labeling
            })

        all_replies = sorted(all_replies, key=lambda x: x["time"]) #sorts by time
        all_replies = all_replies[:100] #take first 100 comments

        #format time strings for csv
        for comment in all_replies:
            comment["time"] = comment["time"].strftime("%Y-%m-%d %H:%M")
        
        #create dataframe and save to tsv
        df = pd.DataFrame(all_replies)
        
        #create filename based on handle and post_id
        filename = f"comments_{handle}_{post_id}.tsv"
        df.to_csv(filename, sep="\t",index=False)
        
        print(f"Saved {len(all_replies)} comments to '{filename}'")
        
    except Exception as e:
        print(f"Error processing {link}: {e}")
        continue

print(f"\n{'='*50}")
print("Processing complete!")
print('='*50)


Processing post 1/5: https://bsky.app/profile/grudgie.bsky.social/post/3mnfglr63322u
Saved 18 comments to 'comments_grudgie.bsky.social_3mnfglr63322u.tsv'

Processing post 2/5: https://bsky.app/profile/politico.com/post/3mnumedpvyk25
Saved 100 comments to 'comments_politico.com_3mnumedpvyk25.tsv'

Processing post 3/5: https://bsky.app/profile/forbes.com/post/3mnulr45aok2q
Saved 85 comments to 'comments_forbes.com_3mnulr45aok2q.tsv'

Processing post 4/5: https://bsky.app/profile/forbes.com/post/3mnutuerepq2o
Saved 100 comments to 'comments_forbes.com_3mnutuerepq2o.tsv'

Processing post 5/5: https://bsky.app/profile/forbes.com/post/3mnt72lrzcy2r
Saved 100 comments to 'comments_forbes.com_3mnt72lrzcy2r.tsv'

Processing complete!
